In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)


In [ ]:

letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    image = images[i].permute(1, 2, 0).numpy()
    plt.imshow(image)
    plt.title(letters[labels[i].item() - 1])
    plt.axis('off')
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
from torchvision import models

model = models.efficientnet_v2_s(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 26)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model.classifier)

In [ ]:
# Write your code here
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), (labels - 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

def validate(model, test_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), (labels - 1).to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


In [ ]:
# Write your code here
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Setup loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# Lists to store metrics
train_losses = []
test_accuracies = []

num_epochs = 10

for epoch in range(num_epochs):
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_accuracy = validate(model, test_loader, criterion, device)

    test_accuracies.append(test_accuracy)

    print(f'Epoch {epoch+1}/{num_epochs}, Test Accuracy: {test_accuracy:.2f}%')


plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), test_accuracies, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy over Epochs')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Write your code here

def validate_tta(model, test_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), (labels - 1).to(device)
            outputs1 = model(images)
            outputs2 = model(torch.flip(images, dims=[3]))
            outputs3 = model(torch.flip(images, dims=[2]))



            avg_outputs = (outputs1 + outputs2 + outputs3) / 3
            _, predicted = torch.max(avg_outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()




    return 100 * correct / total